In [ ]:
# Training configuration — injected at submit time by KaggleTrainingAdapter
import json
CONFIG = {{config}}
print('Config:', json.dumps(CONFIG, indent=2))

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'datasets', 'torch', 'llama-cpp-python', 'pydantic'], check=True)

In [ ]:
import shutil, os
from pathlib import Path

# Kaggle input datasets are mounted at /kaggle/input/<dataset-slug>/
input_dir = Path('/kaggle/input')
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)

# Copy train/eval files from the attached dataset
for candidate in input_dir.rglob('*.jsonl'):
    shutil.copy(candidate, data_dir / candidate.name)
    print(f'Copied {candidate} -> data/{candidate.name}')

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, '-m', 'src.cli.train',
    '--model', CONFIG['model'],
    '--epochs', str(CONFIG['epochs']),
    '--patience', str(CONFIG['patience']),
    '--warmup-ratio', str(CONFIG['warmup_ratio']),
]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, check=True)
print('Training complete.')

In [ ]:
import tarfile
from pathlib import Path

checkpoint_dir = Path('models/checkpoints')
output_archive = Path('/kaggle/working/checkpoint.tar.gz')

with tarfile.open(output_archive, 'w:gz') as tf:
    tf.add(checkpoint_dir, arcname='checkpoints')

print(f'Checkpoint packaged -> {output_archive} ({output_archive.stat().st_size / 1e6:.1f} MB)')